In [6]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler,MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import seaborn as sns



In [7]:
df = sns.load_dataset('tips')
df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   total_bill  244 non-null    float64 
 1   tip         244 non-null    float64 
 2   sex         244 non-null    category
 3   smoker      244 non-null    category
 4   day         244 non-null    category
 5   time        244 non-null    category
 6   size        244 non-null    int64   
dtypes: category(4), float64(2), int64(1)
memory usage: 7.4 KB


In [9]:
num_features = ["total_bill", "size"]
cat_features = ["sex", "smoker", "day", "time"]

num_transformer = Pipeline(steps=[("scaler", StandardScaler())])

cat_transformer = Pipeline(steps=[("encoder", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_transformer, num_features),
        ("cat", cat_transformer, cat_features),
    ]
)

In [10]:
x = df.drop("tip", axis=1)
y = df["tip"]
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

In [13]:
models = {
    "LinearRegression": (LinearRegression(), {}),
    "SVR": (SVR(), {"model__kernel": ["rbf", "poly"]}),
    "DecisionTreeRegressor": (
        DecisionTreeRegressor(),
        {"model__max_depth": [None, 5, 10]},
    ),
    "RandomForestRegressor": (
        RandomForestRegressor(),
        {"model__n_estimators": [10, 100]},
    ),
    "KNeighborsRegressor": (
        KNeighborsRegressor(),
        {"model__n_neighbors": list(range(3, 100, 2))},
    ),
    "GradientBoostingRegressor": (
        GradientBoostingRegressor(),
        {"model__n_estimators": [10, 100]},
    ),
}

best_model = None
best_params = None
best_mae = None
best_mse = None
best_rmse = None
best_r2 = float("-inf")

for name, (model, params) in models.items():

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ]
    )

    grid = GridSearchCV(pipeline, params, cv=5)
    grid.fit(x_train, y_train)

    y_pred = grid.predict(x_test)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    print(f"\n{name}")
    print(f"Best Parameters: {grid.best_params_}")
    print(f"MAE : {mae:.3f}")
    print(f"MSE : {mse:.3f}")
    print(f"RMSE: {rmse:.3f}")
    print(f"R²  : {r2:.3f}")

    # Update best model
    if r2 > best_r2:
        best_model = name
        best_params = grid.best_params_
        best_mae = mae
        best_mse = mse
        best_rmse = rmse
        best_r2 = r2

print("\n" + "=" * 40)
print("BEST MODEL")
print("=" * 40)
print(f"Model           : {best_model}")
print(f"Best Parameters : {best_params}")
print(f"MAE             : {best_mae:.3f}")
print(f"MSE             : {best_mse:.3f}")
print(f"RMSE            : {best_rmse:.3f}")
print(f"R² Score        : {best_r2:.3f}")


LinearRegression
Best Parameters: {}
MAE : 0.667
MSE : 0.703
RMSE: 0.839
R²  : 0.437

SVR
Best Parameters: {'model__kernel': 'rbf'}
MAE : 0.664
MSE : 0.718
RMSE: 0.847
R²  : 0.426

DecisionTreeRegressor
Best Parameters: {'model__max_depth': 5}
MAE : 0.743
MSE : 0.984
RMSE: 0.992
R²  : 0.213

RandomForestRegressor
Best Parameters: {'model__n_estimators': 10}
MAE : 0.770
MSE : 0.942
RMSE: 0.971
R²  : 0.246

KNeighborsRegressor
Best Parameters: {'model__n_neighbors': 5}
MAE : 0.765
MSE : 0.891
RMSE: 0.944
R²  : 0.287

GradientBoostingRegressor
Best Parameters: {'model__n_estimators': 10}
MAE : 0.769
MSE : 0.820
RMSE: 0.905
R²  : 0.344

BEST MODEL
Model           : LinearRegression
Best Parameters : {}
MAE             : 0.667
MSE             : 0.703
RMSE            : 0.839
R² Score        : 0.437
